# Notebook Overview

This Notebook is a continuation from deployment 2 which failed.

# Objective

Consistency in variable naming, interchange dropping operations. Drop users first then books.

# Conclusion

This deployment succeeded ! Major milestones and takeaways were achieved

# Key Takeaways

## Simultenous Dropping Over Sequential dropping

Dropping invalid users and book codes sequentially leads to filtering out of essential entries. Giving wrong information. Instead data should be dropped simultenously checking both conditions at the same time

**Example**

```python

 filtering books with more than 100 ratings

book_counts = filtered_df['isbn'].value_counts() #returns frequencies of unique books

valid_books = book_counts[book_counts >= 100].index

# drops all rows with non valid books

filtered_df = filtered_df[filtered_df['isbn'].isin(valid_books)]

# filtering users with 200+ ratings

user_counts = filtered_df['user'].value_counts()
valid_users = user_counts[user_counts >= 200].index
print(len(valid_users))

# drops all rows with users who are not valid

filtered_df = filtered_df[filtered_df['user'].isin(valid_users)]
```
---

Dropping valid books first excludes non popular books which may have been rated by valid users, therefore reducing their count in the dataset which will cause them to be filtered out in the second threshold
```filtered_df = filtered_df[filtered_df['user'].isin(valid_users)]```. Giving wrong observations

Valid approach is filtering dataset simulteneously

**Example**

```python
filtered_df = filtered_df[filtered_df['user'].isin(valid_users) & filtered_df['isbn'].isin(valid_books)]
```

This approach keeps rows that have both valid users and valid book codes without altering the shape of the dataframe beforehand

---

## Removing Outliers

Outliers are entries that are far away from the spread of the dataset either having extremely low  or very high entries.

**Example**

The Novel the Lovely bones is an outlier as can be seen in this spread

```
[np.int64(1295), np.int64(117), np.int64(309), np.int64(815), np.int64(276), np.int64(585), np.int64(145), np.int64(175)]
```
Most Novels had below 1000 ratings, hence including lovely bones leads to wrong results as it overlaps with many books because of its popularity which might mask real patterns. Therefore it acts as noise

For correct observations I had to filter this kind of books


```python

# remove outliers
# books with more than 1000 ratings and less than 100
remove_outliers= book_counts[(book_counts >= 100) & (book_counts < 1000)].index


# keeps rows where user is valid and also valid book

filtered_df = filtered_df[filtered_df['user'].isin(valid_users) & filtered_df['isbn'].isin(remove_outliers)]

```

# Pandas Merging

Merging the main ratings dataframe and books column correctly filters out books that may have been rated by users but lacked in the main books column

# Pandas pivotting

Correctly sets each unique book code as a row and all columns as unique users
hence forming a vector which maps a unique book to a specific location













In [120]:
# import libraries (you may add additional imports but you may not have to)
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

In [121]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/books/book-crossings.zip

!unzip book-crossings.zip

books_filename = 'BX-Books.csv'
ratings_filename = 'BX-Book-Ratings.csv'

--2026-05-20 16:34:25--  https://cdn.freecodecamp.org/project-data/books/book-crossings.zip
Resolving cdn.freecodecamp.org (cdn.freecodecamp.org)... 172.67.70.149, 104.26.3.33, 104.26.2.33, ...
Connecting to cdn.freecodecamp.org (cdn.freecodecamp.org)|172.67.70.149|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 26085508 (25M) [application/zip]
Saving to: ‘book-crossings.zip.3’

book-crossings.zip. 100%[===================>]  24.88M  --.-KB/s    in 0.1s    

2026-05-20 16:34:25 (226 MB/s) - ‘book-crossings.zip.3’ saved [26085508/26085508]

Archive:  book-crossings.zip
replace BX-Book-Ratings.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: N


In [122]:
# loading books metadata
df_books = pd.read_csv(books_filename,
    encoding = "ISO-8859-1", # latin encoder
    sep = ";",
    header = 0,
    names=['isbn', 'title', 'author'],
    usecols=['isbn', 'title', 'author'], # specifies columns to use
    dtype={'isbn': 'str', 'title': 'str', 'author': 'str'}) # specifies datatype of columns

# loading ratings metadata
df_ratings = pd.read_csv(
    ratings_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['user', 'isbn', 'rating'],
    usecols=['user', 'isbn', 'rating'],
    dtype={'user': 'int32', 'isbn': 'str', 'rating': 'float32'})

In [123]:
df_ratings.head()

,user,isbn,rating
0,276725,034545104X,0.0
1,276726,0155061224,5.0
2,276727,0446520802,0.0
3,276729,052165615X,3.0
4,276729,0521795028,6.0


In [124]:
df_ratings.shape

(1149780, 3)

In [125]:
df_books.head()

,isbn,title,author
0,0195153448,Classical Mythology,Mark P. O. Morford
1,0002005018,Clara Callan,Richard Bruce Wright
2,0060973129,Decision in Normandy,Carlo D'Este
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata
4,0393045218,The Mummies of Urumchi,E. J. W. Barber


In [126]:
df_books.shape

(271379, 3)

In [127]:
# merge titles
# filter book codes that were in df ratings but didnt exist in df books

filtered_df = pd.merge(df_ratings, df_books, on='isbn', how='inner')

# drop authors column
filtered_df = filtered_df.drop(columns = 'author', axis = 1)

#reorganize dataframe columns

filtered_df= filtered_df[['user','isbn','title','rating']]


print(filtered_df.shape)

# rows dropped
print("Number of Rows Dropped after filtering")
print(df_ratings.shape[0] - filtered_df.shape[0])

# NAN Rows
print(f"NaNs per column before: \n{filtered_df.isna().sum()}")

# finding threshold for books

ratings_count = filtered_df['isbn'].value_counts().values

# finding percentile
# sets threshold for filttering books

threshold = np.percentile(ratings_count,99)
print(threshold)



(1031175, 4)
Number of Rows Dropped after filtering
118605
NaNs per column before: 
user      0
isbn      0
title     0
rating    0
dtype: int64
42.0


In [128]:
title_counts = filtered_df['title'].value_counts()


In [129]:
n= [['The Lovely Bones: A Novel', 0.7447561025619507],
  ['The Weight of Water', 0.7660665512084961],
  ['Icy Sparks', 0.7926095128059387],
  ["Bridget Jones's Diary", 0.8025513887405396],
  ['I Know This Much Is True', 0.8076125383377075],["Where the Heart Is (Oprah's Book Club (Paperback))"],["I'll Be Seeing You"],["The Surgeon"]]

codes = [title_counts[title[0]] for title in n]

In [130]:
print(codes)

[np.int64(1295), np.int64(117), np.int64(309), np.int64(815), np.int64(276), np.int64(585), np.int64(145), np.int64(175)]


# conclusion

The lovely bones is an outlier with more than 1000 ratings making it overlap with many books hence its popuylarity pushes it rather than quality of the book

In [131]:
'''
filtering columns
counts are taken from the same dataframe
'''
# filtering users with 200+ ratings

user_counts = filtered_df['user'].value_counts()
valid_users = user_counts[user_counts >= 200].index

# filtering books with more than 100 ratings


book_counts = filtered_df['isbn'].value_counts() #returns frequencies of unique books

# remove outliers
# books with more than 1000 ratings and less than 100
remove_outliers= book_counts[(book_counts >= 100) & (book_counts < 1000)].index


# keeps rows where user is valid and also valid book

filtered_df = filtered_df[filtered_df['user'].isin(valid_users) & filtered_df['isbn'].isin(remove_outliers)]


print("Total number of unique users")
print(filtered_df['user'].nunique())

print("Total Number of invalid users dropped")
print(df_ratings['user'].nunique() - filtered_df['user'].nunique())

print("Total number of unique book codes")
print(filtered_df['isbn'].nunique())

print("Total Number of invalid books dropped")
print(df_books['isbn'].nunique() - filtered_df['isbn'].nunique())


print("Total Number of rows dropped after filtering")
print(df_ratings.shape[0] - filtered_df.shape[0])

print("Final size of dataset")
print(filtered_df.shape)

'''
conversion to categorical data type columns
keeps track of unique values using categorical indexes
hence memory efficient
'''

filtered_df['isbn'] = filtered_df['isbn'].astype('category')
filtered_df['user'] = filtered_df['user'].astype('category')


Total number of unique users
811
Total Number of invalid users dropped
104472
Total number of unique book codes
725
Total Number of invalid books dropped
270654
Total Number of rows dropped after filtering
1102017
Final size of dataset
(47763, 4)


In [132]:
'''
pivotting transforms chosen unique column values to rows, columns and values
Each row is a unique book code with ratings by all unique users
Each column represent all ratings by a single user
csr matrix compresses space
'''

pivot_df = filtered_df.pivot_table(index ='isbn',columns = 'user', values = 'rating',fill_value = 0,observed=True)
sparse_matrix = csr_matrix(pivot_df.values)

# Memory usage
memory_usage_bytes = pivot_df.memory_usage(deep=True).sum()
memory_usage_mb = memory_usage_bytes / (1024 * 1024)
print(f"Memory consumption of pivot_df: {memory_usage_mb:.2f} MB")


# The csr_matrix stores data, indices, and indptr arrays

memory_usage_sparse_bytes = sparse_matrix.data.nbytes + sparse_matrix.indices.nbytes + sparse_matrix.indptr.nbytes
memory_usage_sparse_mb = memory_usage_sparse_bytes / (1024 * 1024)

print(f"Memory consumption of sparse_matrix: {memory_usage_sparse_mb:.2f} MB")

print(f"space saved {memory_usage_mb - memory_usage_sparse_mb}")


Memory consumption of pivot_df: 2.30 MB
Memory consumption of sparse_matrix: 0.09 MB
space saved 2.2067689895629883


In [133]:

'''
Model Creation
brute algorithm for comparing a query to each unique point
'''

from sklearn.neighbors import NearestNeighbors
knn_model = NearestNeighbors(metric='cosine', algorithm='brute')
knn_model.fit(sparse_matrix)


NearestNeighbors(algorithm='brute', metric='cosine')

In [134]:
# getting relevant titles

# filtering dataframe with only relevant book codes and titles

df = df_books[df_books['isbn'].isin(pivot_df.index)]
df.shape


# title to isbn dictionary



titles = df['title'].tolist()
isbn = df['isbn'].tolist()

isbn_title = {code:title for code,title in zip(isbn,titles)}

title_isbn = {title:code for code,title in isbn_title.items()}

In [135]:
from sklearn import neighbors

## function to return recommended books - this will be tested
def get_recommends(book):

 #check if title is valid
  if book not in title_isbn:
    print("Enter valid title")
    return

  # create the first dimension
  recommended_books = [book]

  # second dimension

  neighbors = []

  # get book_code

  book_code = title_isbn[book]

  # get row index for book code

  row_index = pivot_df.index.get_loc(book_code)

  # get csr vector i.e where the book is located

  position_vector = sparse_matrix[row_index]

  # find neighbors

  distances, indices = knn_model.kneighbors(position_vector, n_neighbors= 6)


  # flatten distances and indices from 2D to 1D

  distances, indices = distances.flatten(), indices.flatten()


  for index,row in enumerate(indices):

   # skips the first iteration to avoid recording the query point as a neighbor
    if index == 0:
      continue

    # get book codes of neighbours

    n_code = pivot_df.index[row]

    # neighbors titles

    n_title = isbn_title[n_code]

    # append to second dimension along with their distances as a 3rd dimension(list)

    neighbors.append([n_title, float(distances[index])])


  # append to main title

  recommended_books.append(neighbors)

  return recommended_books

In [136]:
get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")

["Where the Heart Is (Oprah's Book Club (Paperback))",
 [['I Know This Much Is True', 0.7646420001983643],
  ['The Surgeon', 0.7669050693511963],
  ['The Weight of Water', 0.7678344249725342],
  ['Tis: A Memoir', 0.7937096357345581],
  ['Icy Sparks', 0.798843502998352]]]

# Testing function

input - ```get_recommends("The Queen of the Damned (Vampire Chronicles (Paperback))")```

expected_output - ```[
  'The Queen of the Damned (Vampire Chronicles (Paperback))',
  [
    ['Catch 22', 0.793983519077301],
    ['The Witching Hour (Lives of the Mayfair Witches)', 0.7448656558990479],
    ['Interview with the Vampire', 0.7345068454742432],
    ['The Tale of the Body Thief (Vampire Chronicles (Paperback))', 0.5376338362693787],
    ['The Vampire Lestat (Vampire Chronicles, Book II)', 0.5178412199020386]
  ]
]```

In [137]:
# correctly matches but in descending order
get_recommends("The Queen of the Damned (Vampire Chronicles (Paperback))")

['The Queen of the Damned (Vampire Chronicles (Paperback))',
 [['The Vampire Lestat (Vampire Chronicles, Book II)', 0.514513373374939],
  ['The Tale of the Body Thief (Vampire Chronicles (Paperback))',
   0.529854416847229],
  ['Interview with the Vampire', 0.7325513362884521],
  ['The Witching Hour (Lives of the Mayfair Witches)', 0.7362787127494812],
  ['Lasher: Lives of the Mayfair Witches (Lives of the Mayfair Witches)',
   0.7833433151245117]]]

In [138]:
books = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
print(books)

def test_book_recommendation():
  test_pass = True
  recommends = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
  if recommends[0] != "Where the Heart Is (Oprah's Book Club (Paperback))":
    test_pass = False
  recommended_books = ["I'll Be Seeing You", 'The Weight of Water', 'The Surgeon', 'I Know This Much Is True']
  recommended_books_dist = [0.8, 0.77, 0.77, 0.77]
  for i in range(2):
    if recommends[1][i][0] not in recommended_books:
      test_pass = False
    if abs(recommends[1][i][1] - recommended_books_dist[i]) >= 0.05:
      test_pass = False
  if test_pass:
    print("You passed the challenge! 🎉🎉🎉🎉🎉")
  else:
    print("You haven't passed yet. Keep trying!")

test_book_recommendation()

["Where the Heart Is (Oprah's Book Club (Paperback))", [['I Know This Much Is True', 0.7646420001983643], ['The Surgeon', 0.7669050693511963], ['The Weight of Water', 0.7678344249725342], ['Tis: A Memoir', 0.7937096357345581], ['Icy Sparks', 0.798843502998352]]]
You passed the challenge! 🎉🎉🎉🎉🎉
